In [42]:
%reset -f
import sys
for module in list(sys.modules.keys()):
  if module.startswith(("models", "utils", "custom_datasets", "text_classification")):
    del sys.modules[module]
import torch
if torch.cuda.is_available():
  torch.cuda.empty_cache()
!rm -rf /content/*

In [43]:
from torch import nn
import os
from pathlib import Path
device = "cuda" if torch.cuda.is_available() else "cpu"
try:
  from torchinfo import summary
except:
  print("Torchinfo not found! Installing...")
  !pip install -q torchinfo

!git clone https://github.com/asdq11870-cyber/PyTorch
!mv PyTorch/text_classification .
!mv PyTorch/assets .
!mv PyTorch/data/texts .
!mv PyTorch/models .
!mv PyTorch/custom_datasets .
!rm -rf PyTorch

Cloning into 'PyTorch'...
remote: Enumerating objects: 437, done.
remote: Counting objects: 100% (227/227), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 437 (delta 110), reused 145 (delta 56), pack-reused 210 (from 1)
Receiving objects: 100% (437/437), 27.79 MiB | 37.30 MiB/s, done.
Resolving deltas: 100% (235/235), done.
Filtering content: 100% (4/4), 511.73 MiB | 16.80 MiB/s, done.


In [44]:
from models.NanoGPT import GPT
from text_classification.tokenize import NanoGPTTokenizer
from text_classification import utils, data_setup, engine

In [45]:
tokenizer = NanoGPTTokenizer()
with open("texts/input.txt", "r", encoding="utf-8") as f:
  text = f.read()

tokens = tokenizer.encode(text)
vocab_size = tokenizer.encoder.n_vocab
n = len(tokens)
print(f"Total tokens: {n} | vocab_size: {vocab_size}")

Total tokens: 42548582 | vocab_size: 50257


In [46]:
train_tokens = tokens[:int(n*0.9)]
val_tokens = tokens[int(n*0.9):int(n*0.95)]
test_tokens = tokens[int(n*0.95):]

In [53]:
model0 = GPT(vocab_size=vocab_size, embed_dim=768, heads=12, mlp_dim=3072, mlp_dropout=0.1, attn_dropout=0.1, num_encoder_layers=24,
             context_length=256)
model0 = model0.to(device)
model0 = torch.compile(model0)
batch_size = 8
print(batch_size)

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 9.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.03 GiB is allocated by PyTorch, and 391.18 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [51]:
train_dataloader, val_dataloader, test_dataloader = data_setup.create_dataloaders(train_tokens=train_tokens,
                                                                                  val_tokens=val_tokens,
                                                                                  test_tokens=test_tokens,
                                                                                  context_length=model0.context_length,
                                                                                  batch_size=batch_size)

In [52]:
writer0 = utils.create_writer("Text_Classification","NanoGPT","5_epochs")
optimiser0 = torch.optim.AdamW(
    params=model0.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1
)
loss_fn0 = nn.CrossEntropyLoss()
scheduler0 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer=optimiser0, T_max=5
)
engine.batch_train(model0, train_dataloader, val_dataloader, test_dataloader, 5, 1, torch.device(device), False, optimiser0, loss_fn0, writer0, scheduler0,
                   "NanoGPT", "saved_models", vocab_size)

[INFO] Created SummaryWriter, saving to: runs/25-07-2026/Text_Classification/NanoGPT/5_epochs...
Epoch: 1 
 ---------------------------------------------------------


TorchRuntimeError: RuntimeError when making fake tensor call
  Explanation: Dynamo failed to run FX node with fake tensors: call_function <function embedding at 0x7ed443115440>(*(FakeTensor(..., device='cuda:0', size=(s77, 256), dtype=torch.int64), Parameter(FakeTensor(..., size=(50257, 768), requires_grad=True)), None, None, 2.0, False, False), **{}): got RuntimeError('Unhandled FakeTensor Device Propagation for aten.embedding.default, found two different devices cpu, cuda:0')
  Hint: Your code may result in an error when running in eager. Please double check that your code doesn't contain a similar error when actually running eager/uncompiled. You can do this by removing the `torch.compile` call, or by using `torch.compiler.set_stance("force_eager")`. 

  Developer debug context: 

 For more details about this graph break, please visit: https://meta-pytorch.github.io/compile-graph-break-site/gb/gb4315.html

from user code:
   File "/content/models/NanoGPT.py", line 182, in forward
  